<a href="https://colab.research.google.com/github/cubaseuser123/gpt-2-from-scratch/blob/main/gpt2_consolidated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Consolidated Bigram Language Model
This notebook contains the consolidated code from your `gpt2_from_scratch.ipynb`, with all exploratory print statements removed and the hyperparameters and `estimate_loss` function added at the top, matching the porting section of the video.

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 32
# ------------

torch.manual_seed(1337)

#we start with downloading the model first
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

#let's read inside and inspect this dataset
with open("input.txt", "r", encoding="utf-8") as f:
  text = f.read()

#now we separate out all of the unique characters in this dataset
characters = sorted(list(set(text)))
vocab_size = len(characters)

#now let's create tokens out of this, both encoding and decoding. We are doing character level split here, but later we can try out different techniques for tokenization as well.
stoi = {ch: i for i, ch in enumerate(characters)}
itos = {i : ch for i, ch in enumerate(characters)}
encode = lambda s: [stoi[c] for c in s] #we take a string, and output a list of integers.
decode = lambda l: "".join([itos[i] for i in l]) #we take a list of integers and output a string.

#time to tokenize this entire thing. we encode the entire vocabulary into a torch tensor over here
data = torch.tensor(encode(text), dtype=torch.long)

#let's just split the training and testing data right here now.
n = int(0.9 * len(data)) #first 90 percent
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
  #we will generate a small batch of data of inputs x and targets y
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x,y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

#now we have to start coding out a bigram model to run these batch tokens through and get our very first DL model.
class BigramLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embed) #it will go to the lookup table and each token will directly read off logits for the next token
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape
    tok_emb = self.token_embedding_table(idx) #idx and targets are both (B, T) tensor of integers
    pos_emb = self.position_embedding_table(torch.arrange(T, device=device)) # (B, T, C)
    x = tok_emb + pos_emb
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B * T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      #getting the predictions
      logits, loss = self(idx)
      logits = logits[:, -1, :]
      #now we apply softmax
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

model = BigramLanguageModel()
m = model.to(device)

#now we are creating a pytorch adam optimizer for this
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=500)[0].tolist()))


--2026-07-08 16:47:04--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-07-08 16:47:05 (33.4 MB/s) - ‘input.txt.1’ saved [1115394/1115394]

step 0: train loss 4.3886, val loss 4.3734
step 300: train loss 2.5267, val loss 2.5399
step 600: train loss 2.4998, val loss 2.5315
step 900: train loss 2.4903, val loss 2.5085
step 1200: train loss 2.4967, val loss 2.5128
step 1500: train loss 2.4809, val loss 2.5020
step 1800: train loss 2.4858, val loss 2.5149
step 2100: train loss 2.4865, val loss 2.5000
step 2400: train loss 2.4882, val l

From here we start working on the attention mechanism

In [3]:
#There's a mathematical trick in self attention. You just take average of all the past tokens upto and including this token, whilst removing future out of the equation.
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim = True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b
print("a=")
print(a)
print("--")
print("b=")
print(b)
print("--")
print("c=")
print(c)


a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [4]:
#let's consider this sample example
torch.manual_seed(1337)
B,T,C = 4,8,2 #this is batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [6]:
#we need x[b,t]
xbow = torch.zeros((B, T, C))
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] #this is what we were speaking of, from past upto that token, including that token
    xbow[b,t] = torch.mean(xprev, 0)

In [8]:
#we can do a similar thing but now using matrix muliplication
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x #that is (B, T, C) @ (B, T, C) matrix
torch.allclose(xbow, xbow2, atol=1e-06)


True

In [11]:
#we can do a similar thing but this time let's use softmax for it
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3, atol=1e-06)

True

In [12]:
# Let's calculate the maximum absolute difference between xbow and xbow3
max_diff_softmax = torch.max(torch.abs(xbow - xbow3))
print(f"Maximum absolute difference between xbow and xbow3: {max_diff_softmax}")

# Now, let's use a slightly larger atol based on this difference
# A common practice is to use a value slightly larger than the observed max_diff
# For example, if max_diff is 1e-7, you might use 1e-6 or 1e-5
# In this case, 1e-5 should be sufficient
print(f"torch.allclose(xbow, xbow3, atol=1e-05): {torch.allclose(xbow, xbow3, atol=1e-05)}")

Maximum absolute difference between xbow and xbow3: 3.236345946788788e-08
torch.allclose(xbow, xbow3, atol=1e-05): True
